# Camada Gold — Estatística Descritiva e Correlações (Pearson & Spearman)

**Técnicas aplicadas (Aula 07 / Aula 12):**
- **Tendência Central:** Média, Mediana, Moda
- **Dispersão:** Amplitude, Variância, Desvio Padrão, Coeficiente de Variação
- **Posição Relativa:** Quartis (`Q1`, `Q2`, `Q3`), Percentis (`P95`), Decis (`P10`, `P90`)
- **Distribuição de Frequências:** Tabelas de frequência e histogramas categóricos
- **Correlação:** Covariância, Correlação de Pearson e Correlação de Spearman
- **Resumo Estatístico:** `describe()` consolidado

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.gold_utils import (
    compute_descriptive_stats,
    get_gold_paths,
    save_gold_json,
    save_gold_table,
)

ROOT_DIR, SILVER_DIR, GOLD_DIR = get_gold_paths()
print(f"Silver: {SILVER_DIR} | Gold: {GOLD_DIR}")

Silver: C:\Projects\trabalho-final-mineacao-de-dados\data\silver | Gold: C:\Projects\trabalho-final-mineacao-de-dados\data\gold


In [2]:
fato_360 = pd.read_parquet(GOLD_DIR / "fato_aluno_360.parquet")

num_cols = [
    "idade", "renda_estimada", "nota_media", "tempo_logado_medio",
    "paginas_total", "interacoes_total", "valor_mensalidade_total", "taxa_inadimplencia"
]

desc_df = compute_descriptive_stats(fato_360, num_cols)
save_gold_table(desc_df, "gold_estatisticas_descritivas", GOLD_DIR)
print("=== Estatísticas Descritivas ===")
print(desc_df[["variavel", "media", "mediana", "moda", "desvio_padrao", "coef_variacao_pct", "q1_p25", "q3_p75"]].to_string(index=False))

# Correlações (Pearson, Spearman) e Covariância com foco em evasão e desempenho
corr_cols = [
    "evadido_target", "idade", "renda_estimada", "nota_media",
    "tempo_logado_medio", "paginas_total", "interacoes_total", "taxa_inadimplencia"
]
corr_sub = fato_360[corr_cols].dropna()

pearson = corr_sub.corr(method="pearson")
spearman = corr_sub.corr(method="spearman")
cov = corr_sub.cov()

corr_rows = []
for v1 in corr_cols:
    for v2 in corr_cols:
        if v1 < v2:
            corr_rows.append({
                "variavel_1": v1,
                "variavel_2": v2,
                "pearson_r": round(float(pearson.loc[v1, v2]), 4),
                "spearman_rho": round(float(spearman.loc[v1, v2]), 4),
                "covariancia": round(float(cov.loc[v1, v2]), 4),
            })

corr_df = pd.DataFrame(corr_rows).sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)
save_gold_table(corr_df, "gold_matriz_correlacao", GOLD_DIR)
print("\n=== Top Correlações (Pearson & Spearman) ===")
print(corr_df.head(10).to_string(index=False))

# Distribuição de frequências (por faculdade e faixa de renda)
freq_fac = (
    fato_360.groupby("faculdade_principal", as_index=False, observed=True)
    .agg(alunos=("id_aluno", "count"), evadidos=("evadido_target", "sum"), nota_media=("nota_media", "mean"))
)
freq_fac["taxa_evasao_pct"] = (freq_fac["evadidos"] / freq_fac["alunos"] * 100).round(2)
save_gold_table(freq_fac, "gold_freq_faculdade", GOLD_DIR)

=== Estatísticas Descritivas ===
               variavel       media   mediana     moda  desvio_padrao  coef_variacao_pct     q1_p25      q3_p75
                  idade     48.3174     48.00    48.00        18.9482              39.22    32.0000     65.0000
         renda_estimada   7439.4535   6862.96  6862.96      3754.1297              50.46  4367.0200  10068.9700
             nota_media      4.9946      4.99     4.71         2.5824              51.70     3.0000      6.9900
     tempo_logado_medio    249.4804    250.00   263.00       130.1244              52.16   148.0000    350.5000
          paginas_total     20.0389      7.00     0.00        25.7099             128.30     0.0000     36.0000
       interacoes_total    299.6974      0.00     0.00       448.0623             149.50     0.0000    559.0000
valor_mensalidade_total 101857.1130 101568.23 85486.11     13125.1553              12.89 92905.7250 110570.9250
     taxa_inadimplencia      0.5010      0.50     0.50         0.0602  